In [ ]:
import os
import sys
import math
import pickle
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import lr_scheduler

from torchvision import transforms, models, datasets
from torchvision.datasets import ImageFolder
from torchvision.ops import StochasticDepth

import time
import copy

from PIL import Image

In [ ]:
# Paths for logs and models
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

SAVE_DIR = Path('/Users/ductruong/Documents/GitHub/WoodTN/save_model')
SAVE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR = SAVE_DIR / 'runs' / 'effnetv2s_wood60'
LOG_DIR.mkdir(parents=True, exist_ok=True)

writer = SummaryWriter(log_dir=str(LOG_DIR))
print('TensorBoard log dir:', LOG_DIR)


In [ ]:
IMG_SIZE = 224

data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0), ratio=(0.9, 1.1)),
        transforms.RandAugment(num_ops=2, magnitude=10),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.2),
        transforms.ColorJitter(brightness=0.2, contrast=0.25, saturation=0.2, hue=0.02),
        transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), value='random'),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),

    'val': transforms.Compose([
        transforms.Resize(int(IMG_SIZE * 1.15)),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225]),
    ]),
}


In [ ]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

train_root = '/Users/ductruong/Documents/GitHub/WoodTN/Go_that/train'
val_root = '/Users/ductruong/Documents/GitHub/WoodTN/Go_that/val'

trainset = datasets.ImageFolder(train_root, transform=data_transforms['train'])
valset = datasets.ImageFolder(val_root, transform=data_transforms['val'])

# Class-balanced sampler
train_targets = [y for _, y in trainset.samples]
class_counts = Counter(train_targets)
class_weights = {c: 1.0 / max(1, class_counts[c]) for c in class_counts}
sample_weights = [class_weights[y] for y in train_targets]
sampler = WeightedRandomSampler(weights=torch.as_tensor(sample_weights, dtype=torch.double), num_samples=len(sample_weights), replacement=True)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=24,
                                          sampler=sampler, num_workers=4, drop_last=False, pin_memory=True)
valloader = torch.utils.data.DataLoader(valset, batch_size=24,
                                         shuffle=False, num_workers=4, drop_last=False, pin_memory=True)

In [ ]:
dataset_sizes = {"train": len(trainset) ,
                "val": len(valset),
               }
data_loaders = {"train": trainloader,
                "val" : valloader                    
               }

In [ ]:
dataset_sizes

In [ ]:
# EMA helper and warmup cosine scheduler
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.999):
        self.decay = decay
        self.shadow = {name: p.data.clone() for name, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model: nn.Module):
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[name] = (1 - self.decay) * p.data + self.decay * self.shadow[name]
    def apply_to(self, model: nn.Module):
        self.backup = {}
        for name, p in model.named_parameters():
            if p.requires_grad:
                self.backup[name] = p.data.clone()
                p.data = self.shadow[name].clone()
    def restore(self, model: nn.Module):
        for name, p in model.named_parameters():
            if p.requires_grad and name in self.backup:
                p.data = self.backup[name].clone()
        self.backup = {}

# Warmup on top of cosine
class WarmupCosine(torch.optim.lr_scheduler._LRScheduler):
    def __init__(self, optimizer, warmup_epochs, max_epochs, last_epoch=-1):
        self.warmup_epochs = warmup_epochs
        self.max_epochs = max_epochs
        super().__init__(optimizer, last_epoch)
    def get_lr(self):
        if self.last_epoch < self.warmup_epochs:
            warmup_factor = float(self.last_epoch + 1) / float(max(1, self.warmup_epochs))
            return [base_lr * warmup_factor for base_lr in self.base_lrs]
        progress = float(self.last_epoch - self.warmup_epochs) / float(max(1, self.max_epochs - self.warmup_epochs))
        return [0.5 * base_lr * (1.0 + math.cos(math.pi * progress)) for base_lr in self.base_lrs]

In [ ]:
trainset.classes

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
model = models.efficientnet_v2_s(weights=models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(in_features=1280, out_features=len(trainset.classes))
)
# Freeze backbone for warmup
for param in model.features.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True
model = model.to(device, memory_format=torch.channels_last)
use_amp = (device.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
print('Device:', device, 'AMP:', use_amp)

In [ ]:
def mixup_data(x, y, alpha=0.4):
    lam = torch.distributions.Beta(alpha, alpha).sample().item() if alpha > 0 else 1
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    lam = torch.distributions.Beta(alpha, alpha).sample().item()
    b, _, H, W = x.size()
    index = torch.randperm(b, device=x.device)
    y_a, y_b = y, y[index]

    rw = int(W * math.sqrt(1 - lam))
    rh = int(H * math.sqrt(1 - lam))
    cx = torch.randint(0, W, (1,), device=x.device).item()
    cy = torch.randint(0, H, (1,), device=x.device).item()
    x1 = max(cx - rw // 2, 0)
    y1 = max(cy - rh // 2, 0)
    x2 = min(cx + rw // 2, W)
    y2 = min(cy + rh // 2, H)
    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    return x, y_a, y_b, lam

def train_model(model, criterion, optimizer, scheduler, data_loaders, dataset_sizes, device,
                num_epochs, patience=5, use_mixup=True, use_cutmix=True,
                mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_loss = float('inf')
    best_acc = 0.0
    early_stop_counter = 0

    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()

            running_loss = 0.0
            running_corrects = 0
            loop = tqdm(data_loaders[phase], desc=f"{phase} [{epoch+1}/{num_epochs}]")

            for inputs, labels in loop:
                inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                optimizer.zero_grad(set_to_none=True)

                with torch.set_grad_enabled(phase == 'train'):
                    mixed = False
                    if phase == 'train' and torch.rand(1).item() < mix_prob:
                        if use_mixup and use_cutmix:
                            if torch.rand(1).item() < 0.5:
                                inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                            else:
                                inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)
                            mixed = True
                        elif use_mixup:
                            inputs, targets_a, targets_b, lam = mixup_data(inputs, labels, alpha=mixup_alpha)
                            mixed = True
                        elif use_cutmix:
                            inputs, targets_a, targets_b, lam = cutmix_data(inputs, labels, alpha=cutmix_alpha)
                            mixed = True

                    if use_amp and device.type == 'cuda':
                        with torch.cuda.amp.autocast():
                            outputs = model(inputs)
                            if mixed:
                                loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                                acc_targets = targets_a
                            else:
                                loss = criterion(outputs, labels)
                                acc_targets = labels
                        if phase == 'train':
                            scaler.scale(loss).backward()
                            scaler.step(optimizer)
                            scaler.update()
                    else:
                        outputs = model(inputs)
                        if mixed:
                            loss = lam * criterion(outputs, targets_a) + (1 - lam) * criterion(outputs, targets_b)
                            acc_targets = targets_a
                        else:
                            loss = criterion(outputs, labels)
                            acc_targets = labels
                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                running_corrects += torch.sum(preds == acc_targets)

            if phase == 'train' and scheduler is not None:
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val':
                if epoch_loss < best_val_loss:
                    best_val_loss = epoch_loss
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(model.state_dict())
                    early_stop_counter = 0
                    os.makedirs("save_model", exist_ok=True)
                    filename_model = f"save_model/Go_v2_epoch_{epoch}.pth"
                    torch.save(best_model_wts, filename_model)
                    print(f'✅ Model saved: {filename_model}')
                else:
                    early_stop_counter += 1
                    print(f'⏳ No improvement. Early stop counter: {early_stop_counter}/{patience}')

        if early_stop_counter >= patience:
            print(f'⛔ Early stopping at epoch {epoch}')
            break

    time_elapsed = time.time() - since
    print(f'\nTraining complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.nn as nn

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_ft = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=5e-2)
exp_lr_scheduler = WarmupCosine(optimizer_ft, warmup_epochs=5, max_epochs=40)

In [ ]:
model_ft = train_model(model, criterion, optimizer_ft, exp_lr_scheduler,
                      data_loaders, dataset_sizes, device, num_epochs=10, patience=3, use_mixup=True,
                      mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3)

In [ ]:
# Thay X bằng số epoch tốt nhất bạn thấy trong log hoặc file đã lưu
model_ft.load_state_dict(torch.load("save_model/Go_v1_epoch_X.pth"))

In [ ]:
for param in model_ft.parameters():
    param.requires_grad = True

In [ ]:
optimizer_ft = optim.AdamW(model_ft.parameters(), lr=1e-5, weight_decay=1e-4)
exp_lr_scheduler = lr_scheduler.CosineAnnealingLR(optimizer_ft, T_max=30)

In [ ]:
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler,
                      data_loaders, dataset_sizes, device, num_epochs=30, patience=7, use_mixup=True,
                      mixup_alpha=0.4, cutmix_alpha=1.0, mix_prob=0.3)